In [2]:
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
import os

def analyze_cvat_filtered(xml_path, image_folder_path):
    # 1. XML 파일 및 이미지 폴더 확인
    if not os.path.exists(xml_path):
        print(f"❌ 오류: XML 파일 '{xml_path}'을(를) 찾을 수 없습니다.")
        return
    if not os.path.exists(image_folder_path):
        print(f"❌ 오류: 이미지 폴더 '{image_folder_path}'을(를) 찾을 수 없습니다.")
        return

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError as e:
        print(f"❌ XML 파싱 오류: {e}")
        return

    # 2. 변수 초기화
    xml_images = root.findall('image')
    
    valid_image_count = 0      # 실제 존재하는 이미지 수
    missing_image_count = 0    # XML에는 있지만 폴더에 없는 이미지 수
    
    total_boxes = 0
    label_counts = Counter()
    attribute_stats = defaultdict(Counter)

    print("분석 중...", end="\r")

    # 3. 이미지 순회 및 필터링
    for image in xml_images:
        filename = image.get('name')
        full_image_path = os.path.join(image_folder_path, filename)

        # 핵심: 실제 폴더에 파일이 존재하는지 확인
        if os.path.exists(full_image_path):
            valid_image_count += 1
            
            # 박스 통계 집계 (유효한 이미지만)
            boxes = image.findall('box')
            total_boxes += len(boxes)

            for box in boxes:
                label = box.get('label')
                label_counts[label] += 1

                for attr in box.findall('attribute'):
                    attr_name = attr.get('name')
                    attr_value = attr.text
                    attribute_stats[attr_name][attr_value] += 1
        else:
            missing_image_count += 1
            # 필요하다면 아래 주석을 풀어 없는 파일명을 확인할 수 있습니다.
            # print(f"⚠️ 제외됨 (파일 없음): {filename}")

    # 4. 결과 출력
    print("\n" + "=" * 50)
    print(f"📂 XML 파일: {xml_path}")
    print(f"📂 이미지 폴더: {image_folder_path}")
    print("=" * 50)

    print(f"\n1. 🖼️  이미지 현황")
    print(f"   - XML 기록상 총 개수: {len(xml_images)} 장")
    print(f"   - ✅ 실제 존재 (분석 대상): {valid_image_count} 장")
    print(f"   - 🗑️  삭제됨 (분석 제외): {missing_image_count} 장")

    if valid_image_count == 0:
        print("\n❌ 유효한 이미지가 하나도 없습니다. 폴더 경로를 확인해주세요.")
        return

    print(f"\n2. 📦 객체(Bounding Box) 수 (유효 이미지 기준)")
    print(f"   - 총 {total_boxes} 개")
    print(f"   - 이미지 당 평균 {total_boxes / valid_image_count:.2f} 개")

    print(f"\n3. 🏷️  라벨(Class) 분포")
    for label, count in label_counts.items():
        print(f"   - {label}: {count} 개")

    print(f"\n4. 📊 속성(Attribute)별 상세 분포")
    if attribute_stats:
        for attr_name, counts in attribute_stats.items():
            print(f"\n   [ {attr_name} ]")
            for value, count in counts.most_common():
                ratio = (count / total_boxes) * 100
                print(f"     • {value:<10}: {count:>4} 개 ({ratio:.1f}%)")
    else:
        print("   (속성 정보가 없습니다)")
    
    print("\n" + "=" * 50)

# --- 실행 ---
# 1. annotations.xml 파일 경로
# 2. 실제 이미지가 들어있는 폴더 경로 (예: './images' 또는 'C:/my_project/data')
# 경로를 본인의 환경에 맞게 수정해서 실행해 줘.
analyze_cvat_filtered('./dataset_cvat/annotations.xml', './dataset_cvat/images')

분석 중...
📂 XML 파일: ./dataset_cvat/annotations.xml
📂 이미지 폴더: ./dataset_cvat/images

1. 🖼️  이미지 현황
   - XML 기록상 총 개수: 398 장
   - ✅ 실제 존재 (분석 대상): 398 장
   - 🗑️  삭제됨 (분석 제외): 0 장

2. 📦 객체(Bounding Box) 수 (유효 이미지 기준)
   - 총 978 개
   - 이미지 당 평균 2.46 개

3. 🏷️  라벨(Class) 분포
   - ripe_chamoe: 978 개

4. 📊 속성(Attribute)별 상세 분포

   [ occlusion ]
     • visible   :  486 개 (49.7%)
     • partial   :  384 개 (39.3%)
     • severe    :  108 개 (11.0%)

   [ split ]
     • false     :  588 개 (60.1%)
     • true      :  390 개 (39.9%)

   [ truncated ]
     • false     :  834 개 (85.3%)
     • true      :  144 개 (14.7%)

   [ ambiguous ]
     • false     :  930 개 (95.1%)
     • true      :   48 개 (4.9%)



In [3]:
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
import os

def analyze_background_images(xml_path, image_folder_path):
    if not os.path.exists(xml_path):
        print(f"❌ 오류: XML 파일 '{xml_path}'을(를) 찾을 수 없습니다.")
        return
    if not os.path.exists(image_folder_path):
        print(f"❌ 오류: 이미지 폴더 '{image_folder_path}'을(를) 찾을 수 없습니다.")
        return

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError as e:
        print(f"❌ XML 파싱 오류: {e}")
        return

    xml_images = root.findall('image')
    
    valid_image_count = 0      
    empty_image_count = 0      # 객체가 하나도 없는 이미지 수
    empty_image_names = []     # 객체가 없는 이미지 파일명 리스트
    
    print("분석 중...", end="\r")

    for image in xml_images:
        filename = image.get('name')
        full_image_path = os.path.join(image_folder_path, filename)

        # 실제 폴더에 파일이 존재하는 경우만 분석
        if os.path.exists(full_image_path):
            valid_image_count += 1
            
            boxes = image.findall('box')
            
            # 박스가 0개인지 확인
            if len(boxes) == 0:
                empty_image_count += 1
                empty_image_names.append(filename)

    # --- 결과 출력 ---
    print("\n" + "=" * 50)
    print(f"📂 분석 대상 폴더: {image_folder_path}")
    print("=" * 50)

    print(f"\n1. 🖼️  전체 유효 이미지: {valid_image_count} 장")

    if valid_image_count > 0:
        ratio = (empty_image_count / valid_image_count) * 100
        print(f"\n2. 🚫 배경 이미지 (객체 없음): {empty_image_count} 장 ({ratio:.2f}%)")
        print(f"   - 라벨링된 객체가 있는 이미지: {valid_image_count - empty_image_count} 장")
    else:
        print("\n❌ 유효한 이미지가 없습니다.")

    # 배경 이미지 파일명 출력 (너무 많으면 앞 5개만 출력)
    if empty_image_count > 0:
        print("\n3. 📝 배경 이미지 파일명 예시:")
        for name in empty_image_names[:5]:
            print(f"   - {name}")
        if empty_image_count > 5:
            print(f"   ... 외 {empty_image_count - 5}장")
            
    print("\n" + "=" * 50)

# --- 실행 ---
# 경로를 본인 환경에 맞게 수정해 주세요.
analyze_background_images('./dataset_cvat/annotations.xml', './dataset_cvat/images')

분석 중...
📂 분석 대상 폴더: ./dataset_cvat/images

1. 🖼️  전체 유효 이미지: 398 장

2. 🚫 배경 이미지 (객체 없음): 72 장 (18.09%)
   - 라벨링된 객체가 있는 이미지: 326 장

3. 📝 배경 이미지 파일명 예시:
   - 250404_img_046.png
   - 250404_img_047.png
   - 250404_img_048.png
   - 250404_img_057.png
   - 250404_img_097.png
   ... 외 67장

